# 04 機械学習と運用評価

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/ml.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
予測誤差と売買成績の両面からベースラインと比較する。

## 2. 理論と数式
$y_t=P_{t+2}/P_{t+1}-1$。Xはtまでの収益・モメンタム・変動率のみ。

## 3. データの定義
seed42、800営業日×1合成資産。学習60%・検証20%・テスト20%。

## 4. 分析条件・パラメータ
境界2日purge。標準化は訓練だけ。検証RMSEでモデル選択し、テストで再調整しない。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/quant_ai").exists())
os.chdir(ROOT)
(ROOT / "分析結果/図").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("データ種別: 合成データ。実市場での収益性を示す結果ではありません。")


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.ml import make_features
print(inspect.getsourcefile(make_features))
display(Code(inspect.getsource(make_features), language='python'))

In [ ]:
from quant_ai.data_loader import synthetic_prices
from quant_ai.ml import make_features, chronological_split, compare_models
from quant_ai.backtest import backtest
from quant_ai.metrics import performance
prices = synthetic_prices(800, 1)
x, y = make_features(prices.iloc[:, 0])
splits = chronological_split(x, y, gap=2)
scores, predictions = compare_models(splits)
chosen = scores.valid_rmse.idxmin()  # test列はモデル選択に使わない
display(scores)
print("検証期間のみで選択:", chosen)


## 6. 計算結果と可視化

In [ ]:
first, last = predictions[chosen].index[[0, -1]]
# 最後の判断の約定・保有損益まで価格を残す
end_pos = prices.index.get_loc(last)+2
test_prices = prices.loc[first:prices.index[end_pos]]
navs, stats = {}, {}
for name in dict.fromkeys([chosen, "zero", "historical_mean", "lightgbm"]):
    pred = predictions[name]
    signals = pd.DataFrame(0., index=test_prices.index, columns=prices.columns)
    signals.loc[pred.index, prices.columns[0]] = (pred > 0).astype(float)
    # 最終予測後も最終目標を維持（新情報は使わない）
    signals.loc[signals.index > last] = signals.loc[last].to_numpy()
    bt = backtest(test_prices, signals, cost_bps=5, benchmark=test_prices.iloc[:, 0])
    navs[name] = bt.wealth
    stats[name] = performance(bt.net_return)
stats["buy_hold_gross"] = performance(bt.benchmark_return)
display(pd.DataFrame(stats))
ax = pd.DataFrame(navs).plot(title="Frozen model signals - held-out synthetic period")
ax.figure.savefig("分析結果/図/04_機械学習の運用比較.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 統計的解釈
線形/Ridge/Lasso/RF/LightGBMをゼロ予測・過去平均と比較する。低RMSEと高利益は別。

## 8. 金融工学・運用上の解釈
判断翌終値の約定とコストを含める。合成市場に収益機会を意図して入れていない。

## 9. 限界と追加検証
## 解釈
RMSEの優劣と売買成績の優劣は一致しません。ゼロ予測戦略は現金です。
コスト控除後のSharpe・最大DD・CAGRでリスクも比較します。合成市場には予測可能性を
意図的に組み込んでいないため、複雑なモデルが勝つことは期待していません。

## 限界・改善
単一の分割、固定特徴量、閾値0、売買可能性の単純化があります。
テストを繰り返し見ればモデル選択バイアスが生じます。ウォークフォワード、
ハイパーパラメータの訓練内CV、ブロックブートストラップ、独立期間での再評価へ進みます。